In [1]:
#we will create two preprocessing pipelines that will be used in training 
# and when they are fited, in production, this is step one of getting the model ready for prod
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin, OneToOneFeatureMixin
from sklearn.impute import SimpleImputer
import dill

In [12]:
from risk_scorer_runtime import lowercase_column, replace_rare_values_transformer

In [13]:
from risk_scorer_runtime import FeatureMultiplier

In [14]:
from risk_scorer_runtime import FeatureDivider

In [15]:
coverage_type_pipeline_catboost = Pipeline([
    ('lowercase', FunctionTransformer(lowercase_column)),
    ('filter_values', FunctionTransformer(
        replace_rare_values_transformer,
        kw_args={
            'allowed_values': ['al', 'apd'],
            'default_value': 'other'
        }
    ))
])

business_type_pipeline_catboost = Pipeline([
    ('lowercase', FunctionTransformer(lowercase_column)),
    ('filter_values', FunctionTransformer(
        replace_rare_values_transformer,
        kw_args={
            'allowed_values': ['service', 'trucking', 'construction', 'retail', 'other'],
            'default_value': 'other'
        }
    ))
])

state_pipeline_catboost = Pipeline([
    ('lowercase', FunctionTransformer(lowercase_column)),
    ('filter_values', FunctionTransformer(
        replace_rare_values_transformer,
        kw_args={
            'allowed_values': ['az', 'tx', 'ny', 'il', 'fl', 'oh', 'ga', 'wa', 'nc', 'co', 'ca', 'pa'],
            'default_value': 'other'
        }
    ))
])

preprocessor_cat = ColumnTransformer(
    transformers=[
        ('coverage_preprocessor', coverage_type_pipeline_catboost, ['coverage_type']),
        ('btype_preprocessor', business_type_pipeline_catboost, ['business_type']),
        ('state_preprocessor', state_pipeline_catboost, ['state']),
        ('multiplier_vehicle_age', FeatureMultiplier(
            col1='vehicle_avg_age', col2='vehicle_count', new_col_name='vehicle_age_sum'
        ), ['vehicle_avg_age', 'vehicle_count']),
        ('multiplier_driver_age', FeatureMultiplier(
            col1='driver_avg_age', col2='driver_count', new_col_name='driver_age_sum'
        ), ['driver_avg_age', 'driver_count']),
        ('divider_miles_al', FeatureDivider(
            numerator='prior_year_mileage_000', denominator='prior_al_claim_count',
            new_col_name='miles_al_ratio'
        ), ['prior_year_mileage_000', 'prior_al_claim_count']),
        ('divider_miles_apd', FeatureDivider(
            numerator='prior_year_mileage_000', denominator='prior_apd_claim_count',
            new_col_name='miles_apd_ratio'
        ), ['prior_year_mileage_000', 'prior_apd_claim_count']),
        ('keep', 'passthrough', [
            'years_in_business', 'num_heavy_vehicles', 'prior_year_mileage_000',
            'prior_apd_claim_count', 'prior_al_claim_count', 'prior_loss_amount'
        ])
    ],
    remainder='drop'
)

pipeline_prep_catboost_v2 = Pipeline([
    ('preprocessor_catboost', preprocessor_cat)
])

In [16]:
coverage_type_pipeline = Pipeline([
    ('lowercase', FunctionTransformer(lowercase_column)),
    ('filter_values', FunctionTransformer(
        replace_rare_values_transformer,
        kw_args={'allowed_values': ['al', 'apd'], 'default_value': 'other'}
    )),
    ('one_hot', OneHotEncoder(handle_unknown='infrequent_if_exist', drop='first', sparse_output=False))
])

business_type_pipeline = Pipeline([
    ('lowercase', FunctionTransformer(lowercase_column)),
    ('filter_values', FunctionTransformer(
        replace_rare_values_transformer,
        kw_args={
            'allowed_values': ['service', 'trucking', 'construction', 'retail', 'other'],
            'default_value': 'other'
        }
    )),
    ('one_hot', OneHotEncoder(handle_unknown='infrequent_if_exist', drop='first', sparse_output=False))
])

state_pipeline = Pipeline([
    ('lowercase', FunctionTransformer(lowercase_column)),
    ('filter_values', FunctionTransformer(
        replace_rare_values_transformer,
        kw_args={
            'allowed_values': ['az', 'tx', 'ny', 'il', 'fl', 'oh', 'ga', 'wa', 'nc', 'co', 'ca', 'pa'],
            'default_value': 'other'
        }
    )),
    ('one_hot', OneHotEncoder(handle_unknown='infrequent_if_exist', drop='first', sparse_output=False))
])

def scale_engineered_feature(transformer):
    return Pipeline([
        ('feature', transformer),
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

preprocessor_log = ColumnTransformer(
    transformers=[
        ('coverage_preprocessor', coverage_type_pipeline, ['coverage_type']),
        ('btype_preprocessor', business_type_pipeline, ['business_type']),
        ('state_preprocessor', state_pipeline, ['state']),
        ('multiplier_vehicle_age', scale_engineered_feature(FeatureMultiplier(
            col1='vehicle_avg_age', col2='vehicle_count', new_col_name='vehicle_age_sum'
        )), ['vehicle_avg_age', 'vehicle_count']),
        ('multiplier_driver_age', scale_engineered_feature(FeatureMultiplier(
            col1='driver_avg_age', col2='driver_count', new_col_name='driver_age_sum'
        )), ['driver_avg_age', 'driver_count']),
        ('divider_miles_al', scale_engineered_feature(FeatureDivider(
            numerator='prior_year_mileage_000', denominator='prior_al_claim_count',
            new_col_name='miles_al_ratio'
        )), ['prior_year_mileage_000', 'prior_al_claim_count']),
        ('divider_miles_apd', scale_engineered_feature(FeatureDivider(
            numerator='prior_year_mileage_000', denominator='prior_apd_claim_count',
            new_col_name='miles_apd_ratio'
        )), ['prior_year_mileage_000', 'prior_apd_claim_count']),
        ('numeric_pipeline', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), [
            'years_in_business', 'prior_year_mileage_000', 'prior_apd_claim_count',
            'prior_al_claim_count', 'prior_loss_amount', 'num_heavy_vehicles'
        ])
    ],
    remainder='drop'
)

pipeline_prep_lg_v2 = Pipeline([
    ('preprocessor_logr', preprocessor_log)
])

In [17]:
with open('pipeline_prep_catboost_v2.dill', 'wb') as f:
    dill.dump(pipeline_prep_catboost_v2, f)

with open('pipeline_prep_lg_v2.dill', 'wb') as f:
    dill.dump(pipeline_prep_lg_v2, f)

In [18]:
#just for testing our pipelines, we will be saving them on the model script
from sklearn.model_selection import train_test_split
df_train_auto_policies = pd.read_csv("../data/train.csv")


df_train_auto_policies["had_claim_binary"] = (df_train_auto_policies["claim_count"] > 0).astype(int)
df_train_auto_policies['avg_loss'] = np.nan
claim_mask = df_train_auto_policies['claim_count'] > 0
df_train_auto_policies.loc[claim_mask, 'avg_loss'] = (
    df_train_auto_policies.loc[claim_mask, 'total_loss_amount']
    / df_train_auto_policies.loc[claim_mask, 'claim_count']
)

feat_cols = [
    "coverage_type",
    "vehicle_count", "vehicle_avg_age", "driver_count", "driver_avg_age",
    "years_in_business", "prior_year_mileage_000",
    "business_type", "state",
    "prior_apd_claim_count", "prior_al_claim_count", "prior_loss_amount", "num_heavy_vehicles"
]
targets = ["had_claim_binary", "claim_count", "avg_loss"]


X = df_train_auto_policies[feat_cols]
y = df_train_auto_policies[targets]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y['had_claim_binary'])

with open('pipeline_prep_lg_v2.dill', 'rb') as f:
    loaded_cleaning_pipe_lg_v2 = dill.load(f)

with open('pipeline_prep_catboost_v2.dill', 'rb') as f:
    loaded_cleaning_pipe_cat_v2 = dill.load(f)

loaded_cleaning_pipe_lg_v2.set_output(transform="pandas")
loaded_cleaning_pipe_cat_v2.set_output(transform="pandas")

X_train_lg_ready = loaded_cleaning_pipe_lg_v2.fit_transform(X_train)

In [9]:
X_train_cat_ready = loaded_cleaning_pipe_cat_v2.fit_transform(X_train)

In [10]:
X_train_lg_ready

,coverage_preprocessor__coverage_type_apd,btype_preprocessor__business_type_other,btype_preprocessor__business_type_retail,btype_preprocessor__business_type_service,btype_preprocessor__business_type_trucking,state_preprocessor__state_ca,state_preprocessor__state_co,state_preprocessor__state_fl,state_preprocessor__state_ga,state_preprocessor__state_il,...,multiplier_vehicle_age__vehicle_age_sum,multiplier_driver_age__driver_age_sum,divider_miles_al__miles_al_ratio,divider_miles_apd__miles_apd_ratio,numeric_pipeline__years_in_business,numeric_pipeline__prior_year_mileage_000,numeric_pipeline__prior_apd_claim_count,numeric_pipeline__prior_al_claim_count,numeric_pipeline__prior_loss_amount,numeric_pipeline__num_heavy_vehicles
6160,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-0.288780,0.213778,-0.990071,-1.091775,-0.447242,-1.143920,0.520149,-0.532187,0.661500,-1.273995
7052,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,-0.594081,-0.915916,-0.609633,0.401526,-0.486588,-0.055802,-0.729701,1.242260,-0.407324,-1.273995
4403,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.415761,-0.972667,-0.603457,0.414812,0.090486,-0.044404,-0.729701,1.242260,-0.275282,1.897842
11493,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.484315,2.095419,1.758635,0.386469,-0.906277,1.392488,0.520149,-0.532187,-0.238589,-1.273995
9281,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,3.038219,-0.117859,0.364519,-0.681102,1.375786,0.106048,1.770000,-0.532187,0.539356,-1.273995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11034,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.016521,-0.163969,-0.310718,-0.136098,0.247869,-0.517036,-0.729701,-0.532187,-0.530697,0.311924
10919,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.934609,-1.043605,-0.448235,-0.284011,0.090486,-0.643933,-0.729701,-0.532187,-0.530697,1.104883
14386,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.227884,-0.509794,-0.298366,-0.122812,0.379023,-0.505638,-0.729701,-0.532187,-0.530697,0.311924
6467,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.241640,1.964183,-0.599752,-0.446980,-0.880047,-0.783747,-0.729701,-0.532187,-0.530697,0.311924


In [11]:
X_train_cat_ready

,coverage_preprocessor__coverage_type,btype_preprocessor__business_type,state_preprocessor__state,multiplier_vehicle_age__vehicle_age_sum,multiplier_driver_age__driver_age_sum,divider_miles_al__miles_al_ratio,divider_miles_apd__miles_apd_ratio,keep__years_in_business,keep__num_heavy_vehicles,keep__prior_year_mileage_000,keep__prior_apd_claim_count,keep__prior_al_claim_count,keep__prior_loss_amount
6160,al,trucking,ny,33.0,276.6,50.80,25.400000,4.6,0,50.8,1,0,21424.13
7052,al,retail,ca,25.2,149.2,97.00,194.000000,4.3,0,194.0,0,1,2217.05
4403,al,service,az,51.0,142.8,97.75,195.500000,8.7,4,195.5,0,1,4589.88
11493,al,retail,pa,78.3,488.8,384.60,192.300000,1.1,0,384.6,1,0,5249.27
9281,apd,trucking,ga,118.0,NaN,215.30,71.766667,18.5,0,215.3,2,0,19229.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11034,apd,other,az,40.8,234.0,133.30,133.300000,9.9,2,133.3,0,0,0.00
10919,apd,construction,az,16.5,134.8,116.60,116.600000,8.7,3,116.6,0,0,0.00
14386,al,construction,il,46.2,195.0,134.80,134.800000,10.9,2,134.8,0,0,0.00
6467,apd,trucking,nc,72.1,474.0,98.20,98.200000,1.3,2,98.2,0,0,0.00
